# Módulo I1 · Soluciones de los ejercicios

**Datos no estructurados: clustering y reducción de dimensión**

Este notebook es **autocontenido**: reproduce la preparación mínima del notebook del módulo (datos RFM simulados con la misma
semilla, escalado, k-means con *k* = 4, dígitos y t-SNE) y resuelve los cuatro ejercicios finales.

> Nota para instructores: los números exactos dependen de la versión de scikit-learn; las conclusiones no deberían cambiar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.datasets import load_digits

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(0)

## Preparación (igual que en el notebook)

In [ ]:
perfiles = {
    #               recencia(días) frecuencia(compras/año) gasto_medio(€)
    "fieles":       (15,  25, 60),
    "ocasionales":  (90,   5, 45),
    "premium":      (30,  10, 250),
    "dormidos":     (300,  2, 35),
}
filas = []
for nombre, (r, f, m) in perfiles.items():
    k = 375
    filas.append(pd.DataFrame({
        "recencia": rng.gamma(4, r / 4, k),
        "frecuencia": rng.poisson(f, k) + 1,
        "gasto_medio": rng.normal(m, m * 0.2, k).clip(5),
        "perfil_real": nombre,
    }))
clientes = pd.concat(filas, ignore_index=True)
variables = ["recencia", "frecuencia", "gasto_medio"]
X = clientes[variables]
X_esc = StandardScaler().fit_transform(X)

km4 = KMeans(4, n_init=10, random_state=0).fit(X_esc)
clientes["segmento"] = km4.labels_
print("ARI k=4 (escalado):", round(adjusted_rand_score(clientes["perfil_real"], km4.labels_), 2))

---
## Ejercicio 1 · Nombre de negocio y acción para cada segmento

Primero miramos el perfil medio de cada segmento **en unidades originales** (días, compras/año, euros), su tamaño y cuánto
aporta cada uno al gasto anual estimado (frecuencia × gasto medio). Para que la solución no dependa del número que k-means
asigne a cada grupo, ponemos los nombres con reglas sencillas sobre las medias.

In [ ]:
clientes["valor_anual"] = clientes["frecuencia"] * clientes["gasto_medio"]
resumen = clientes.groupby("segmento").agg(
    recencia=("recencia", "mean"),
    frecuencia=("frecuencia", "mean"),
    gasto_medio=("gasto_medio", "mean"),
    valor_anual=("valor_anual", "mean"),
    clientes=("recencia", "size"),
)
resumen["% clientes"] = 100 * resumen["clientes"] / resumen["clientes"].sum()
resumen["% valor"] = 100 * clientes.groupby("segmento")["valor_anual"].sum() / clientes["valor_anual"].sum()


def nombrar(fila, tabla):
    """Asigna un nombre de negocio a partir de las medias del segmento."""
    if fila.name == tabla["gasto_medio"].idxmax():
        return "VIP de ticket alto"
    if fila.name == tabla["frecuencia"].idxmax():
        return "Fieles recurrentes"
    if fila.name == tabla["recencia"].idxmax():
        return "Dormidos / en riesgo"
    return "Ocasionales"


resumen["nombre"] = resumen.apply(nombrar, axis=1, tabla=resumen)
acciones = {
    "VIP de ticket alto": "Programa VIP: acceso anticipado a novedades, gestor personal y envío gratis; evitar descuentos agresivos.",
    "Fieles recurrentes": "Programa de puntos y venta cruzada (subir el ticket medio con packs o productos complementarios).",
    "Ocasionales": "Recordatorios por email según el ciclo de compra y un incentivo para la 2.ª compra del trimestre.",
    "Dormidos / en riesgo": "Campaña de reactivación ('te echamos de menos') con cupón limitado; si no responden, reducir presión comercial.",
}
resumen["acción"] = resumen["nombre"].map(acciones)
pd.set_option("display.max_colwidth", 120)
resumen.round(1)

In [ ]:
pd.crosstab(clientes["segmento"].map(resumen["nombre"]), clientes["perfil_real"])

**Interpretación (respuesta modelo).**

| Segmento | Perfil medio (aprox.) | Acción |
|---|---|---|
| **VIP de ticket alto** | ~30 días desde la última compra, ~11 compras/año, ~250 € por pedido. ~24 % de los clientes pero **más de la mitad del valor**. | Cuidar la relación (servicio, exclusividad), no regalar margen con descuentos. |
| **Fieles recurrentes** | Compran cada pocas semanas (~26 compras/año), ticket ~60 €. | Fidelización y venta cruzada para subir el ticket. |
| **Ocasionales** | ~115 días de recencia, ~5 compras/año, ~43 €. Es el grupo más grande. | Recordatorios y empujón a la siguiente compra. |
| **Dormidos / en riesgo** | Más de 400 días sin comprar, ~3 compras/año, ~35 €. | Reactivación barata y medible; si no responden, dejar de invertir. |

Observa la tabla cruzada: el segmento «Ocasionales» **mezcla** los ocasionales reales con una parte de los dormidos (los que
llevan "solo" 100–250 días sin comprar). Es un aviso útil para el informe: la frontera entre «ocasional» y «dormido» es
difusa, así que conviene combinar el segmento con una regla de negocio (p. ej., recencia > 180 días → reactivación).

---
## Ejercicio 2 · k = 3 y k = 6: ¿qué se fusiona y qué se divide?

In [ ]:
etiquetas = {}
for k in [3, 4, 6]:
    modelo = KMeans(k, n_init=10, random_state=0).fit(X_esc)
    etiquetas[k] = modelo.labels_
    print(f"k={k}: silueta={silhouette_score(X_esc, modelo.labels_):.3f} · "
          f"ARI vs perfiles reales={adjusted_rand_score(clientes['perfil_real'], modelo.labels_):.2f}")

nombres4 = clientes["segmento"].map(resumen["nombre"])

### k = 3: ¿qué segmentos de k = 4 se fusionan?

In [ ]:
pd.crosstab(pd.Series(etiquetas[3], name="cluster k=3"), nombres4.rename("segmento k=4"))

### k = 6: ¿qué segmentos de k = 4 se dividen?

In [ ]:
tabla6 = clientes.assign(cluster_k6=etiquetas[6]).groupby("cluster_k6")[variables].mean().round(1)
tabla6["clientes"] = pd.Series(etiquetas[6]).value_counts().sort_index()
display(pd.crosstab(pd.Series(etiquetas[6], name="cluster k=6"), nombres4.rename("segmento k=4")))
tabla6

**Interpretación.**

- Con **k = 3**, k-means junta los dos grupos de ticket bajo y baja frecuencia: «Ocasionales» y «Dormidos» pasan a ser un único
  segmento (fíjate en la columna de la tabla cruzada donde se concentran ambos). Se pierde justo la distinción más accionable
  para marketing: a quién hay que reactivar.
- Con **k = 6**, k-means trocea segmentos que ya eran coherentes: los **VIP** se parten en dos según el ticket (≈ 210 € frente a
  ≈ 290 €, con la misma recencia y frecuencia), y la franja ocasional–dormido se divide en tres tramos de recencia (≈ 90, ≈ 255 y
  ≈ 485 días). Las diferencias entre subgrupos son de grado, no de naturaleza, y la silueta baja (≈ 0,54 frente a ≈ 0,64 con k = 4).
  Lo único aprovechable es el tramo intermedio de ≈ 255 días: clientes «enfriándose» a los que aún se puede reactivar.
- **Para Dirección presentaría k = 4**: tiene la mejor silueta, cada segmento tiene un nombre y una acción distinta, y es un número
  que un equipo de marketing puede gestionar. k = 6 tendría sentido solo si existieran seis campañas diferentes que lanzar; si
  no, añade complejidad sin cambiar decisiones.

---
## Ejercicio 3 · k-means sobre píxeles frente a k-means sobre t-SNE (dígitos)

In [ ]:
digits = load_digits()
km_pix = KMeans(10, n_init=10, random_state=0).fit(digits.data)
ari_pix = adjusted_rand_score(digits.target, km_pix.labels_)

Z_tsne = TSNE(2, perplexity=30, random_state=0).fit_transform(digits.data)
km_tsne = KMeans(10, n_init=10, random_state=0).fit(Z_tsne)
ari_tsne = adjusted_rand_score(digits.target, km_tsne.labels_)

print(f"ARI k-means sobre 64 píxeles : {ari_pix:.2f}")
print(f"ARI k-means sobre 2D de t-SNE: {ari_tsne:.2f}")

¿Es estable el resultado sobre t-SNE? Repetimos t-SNE con otras semillas y otra *perplexity*: si los clusters dependen de esas
decisiones arbitrarias, no son una propiedad de los datos.

In [ ]:
filas_estab = []
for semilla in [0, 1]:
    for perp in [5, 30, 50]:
        Zt = TSNE(2, perplexity=perp, random_state=semilla).fit_transform(digits.data)
        lab = KMeans(10, n_init=10, random_state=0).fit_predict(Zt)
        filas_estab.append({"semilla": semilla, "perplexity": perp,
                            "ARI vs dígito": adjusted_rand_score(digits.target, lab),
                            "_labels": lab})
estab = pd.DataFrame(filas_estab)
ref = estab.loc[(estab.semilla == 0) & (estab.perplexity == 30), "_labels"].iloc[0]
estab["ARI vs ejecución de referencia"] = [adjusted_rand_score(ref, l) for l in estab["_labels"]]
estab.drop(columns="_labels").round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(*Z_tsne.T, c=digits.target, cmap="tab10", s=6)
axes[0].set_title("t-SNE coloreado por dígito real")
axes[1].scatter(*Z_tsne.T, c=km_tsne.labels_, cmap="tab10", s=6)
axes[1].set_title(f"t-SNE coloreado por k-means (k=10) · ARI={ari_tsne:.2f}")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

El ARI alto sobre t-SNE parece una buena noticia. Pero ¿qué pasa si aplicamos la misma receta a datos **sin ningún grupo**?
Generamos 500 puntos de ruido gaussiano en 10 dimensiones, los proyectamos con t-SNE y hacemos k-means sobre el mapa.

In [ ]:
ruido = np.random.default_rng(1).normal(size=(500, 10))
Z_ruido = TSNE(2, perplexity=5, random_state=0).fit_transform(ruido)
lab_ruido = KMeans(6, n_init=10, random_state=0).fit_predict(Z_ruido)
print(f"Silueta medida en el mapa t-SNE      : {silhouette_score(Z_ruido, lab_ruido):.2f}")
print(f"Silueta de esos grupos en los datos  : {silhouette_score(ruido, lab_ruido):.2f}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(*Z_ruido.T, c=lab_ruido, cmap="tab10", s=8)
ax.set_title("t-SNE (perplexity=5) de ruido puro + k-means")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Interpretación.**

- Sobre los 64 píxeles, k-means alcanza un ARI moderado (≈ 0,67): algunos dígitos (0, 6) forman grupos compactos, pero otros
  (1/7, 3/5/8/9) se solapan en el espacio original y k-means, que busca grupos esféricos, los mezcla.
- Sobre las 2 coordenadas de t-SNE el ARI **sube** (≈ 0,88): t-SNE ya ha hecho el trabajo de separar vecindades y k-means solo
  tiene que "cortar" islas bien definidas. En este conjunto, tan limpio, el resultado es bastante estable entre semillas
  (scikit-learn inicializa t-SNE con PCA), pero **depende de la perplexity**: con perplexity = 5 la partición ya no coincide con la
  de referencia (ARI ≈ 0,86 entre ambas) y el acuerdo con los dígitos baja a ≈ 0,84.
- **¿Por qué es peligroso?** El experimento con ruido lo muestra: en datos **sin estructura**, el mapa t-SNE presenta "grumos"
  y k-means sobre el mapa obtiene una silueta aceptable (≈ 0,37), mientras que esos mismos grupos en los datos originales no
  tienen ninguna cohesión (silueta ≈ 0,02). t-SNE **no conserva distancias ni densidades globales**, puede exagerar separaciones,
  romper un grupo real en varias islas o fabricar grupos a partir de ruido. Además no tiene `transform`: no podrías asignar un
  segmento a un cliente nuevo sin recalcular todo el mapa.
- Regla práctica: usa t-SNE para **mirar** y generar hipótesis; haz el clustering sobre los datos (o sobre PCA) y valida con
  métricas calculadas en el espacio original y con pruebas de estabilidad.

---
## Ejercicio 4 · Añadir la variable «antigüedad del cliente»

El enunciado no fija cómo se genera la antigüedad, así que probamos **dos escenarios** que se dan en la realidad:

- **(a) Antigüedad informativa**: se relaciona con el perfil (los fieles y los dormidos llevan más tiempo con nosotros; los
  premium son más recientes).
- **(b) Antigüedad sin relación con el perfil**: ruido respecto a la estructura de comportamiento.

In [ ]:
rng4 = np.random.default_rng(42)
antig_media = {"fieles": 48, "ocasionales": 24, "premium": 12, "dormidos": 40}  # meses
clientes["antiguedad_inf"] = [rng4.gamma(6, antig_media[p] / 6) for p in clientes["perfil_real"]]
clientes["antiguedad_ruido"] = rng4.uniform(1, 72, len(clientes))
# Extensión: ¿y si añadimos varias variables irrelevantes a la vez?
for j in range(1, 4):
    clientes[f"irrelevante_{j}"] = rng4.uniform(0, 1, len(clientes))

resultados = []
for nombre, cols in [("solo RFM", variables),
                     ("RFM + antigüedad informativa", variables + ["antiguedad_inf"]),
                     ("RFM + antigüedad sin relación", variables + ["antiguedad_ruido"]),
                     ("RFM + 3 variables irrelevantes", variables + ["irrelevante_1", "irrelevante_2", "irrelevante_3"])]:
    Xe = StandardScaler().fit_transform(clientes[cols])
    lab = KMeans(4, n_init=10, random_state=0).fit_predict(Xe)
    clientes[f"seg_{len(resultados)}"] = lab
    sil_k = {kk: silhouette_score(Xe, KMeans(kk, n_init=10, random_state=0).fit_predict(Xe)) for kk in range(2, 8)}
    resultados.append({"variables": nombre,
                       "k con mejor silueta": max(sil_k, key=sil_k.get),
                       "silueta (k=4)": silhouette_score(Xe, lab),
                       "ARI vs perfiles reales": adjusted_rand_score(clientes["perfil_real"], lab),
                       "ARI vs segmentos originales": adjusted_rand_score(clientes["segmento"], lab)})
pd.DataFrame(resultados).round(2)

In [ ]:
for i, titulo in [(1, "con antigüedad informativa"), (2, "con antigüedad sin relación")]:
    col_ant = "antiguedad_inf" if i == 1 else "antiguedad_ruido"
    perfil = clientes.groupby(f"seg_{i}")[variables + [col_ant]].mean().round(1)
    perfil["clientes"] = clientes[f"seg_{i}"].value_counts().sort_index()
    print(f"\nSegmentos {titulo}:")
    display(perfil)

**Interpretación.**

- **(a) Antigüedad informativa:** los segmentos se mantienen en esencia (ARI ≈ 0,90 frente a los originales) y el acuerdo con
  los perfiles reales mejora ligeramente (≈ 0,76 → ≈ 0,78): la antigüedad ayuda un poco a separar dormidos (veteranos) de
  ocasionales, y el segmento «Dormidos» gana unos 30 clientes. La silueta baja (≈ 0,64 → ≈ 0,53) porque la antigüedad tiene
  mucha dispersión dentro de cada perfil: los grupos son más "anchos" en esa dimensión. Ojo: **no compares siluetas calculadas
  con conjuntos de variables distintos** como si midieran lo mismo.
- **(b) Antigüedad sin relación:** los segmentos apenas cambian (ARI ≈ 0,97 frente a los originales; la estructura RFM es muy
  fuerte), pero la silueta cae a ≈ 0,42. Con **3 variables irrelevantes** cae a ≈ 0,24: k-means pondera **igual** todas las
  variables escaladas, así que el ruido "diluye" las distancias y la evidencia a favor de los grupos se debilita. Además cambia
  el *k* que elegirías mirando solo la silueta: en las tres variantes ampliadas el máximo pasa de k = 4 a k = 3 (columna «k con
  mejor silueta»), que fusiona ocasionales y dormidos. Con una estructura más débil que la de este ejemplo, los propios
  segmentos se degradarían.
- **Lección:** añadir variables no es gratis en clustering. Antes de incluir una pregúntate si representa algo que deba definir
  el segmento. Alternativas: usar la antigüedad **para describir** los segmentos (perfilado a posteriori) en lugar de para
  formarlos, o darle menos peso.